## Karthik Aggarwal - Final Project

**Pitch:**

In September 2026, a cluster of Legionnaires' disease in the South Bronx neighborhoods of Melrose and Morrisania sickened 24 people and killed two, according to the Dept. of Health. It followed an Upper East Side outbreak in July with 11 deaths and a Central Harlem outbreak in July-August 2025 that killed seven. Three outbreaks in roughly 14 months raise a question about the regulatory system meant to prevent them: How consistently are the  City's cooling towers following inspection rules, and crucially, does that differ by borough?

Using cooling tower inspection data from NYC Open Data, I am pitching a data investigation into whether the Bronx has kept pace with Manhattan in complying with the DOH Legionnaires' disease prevention regulations. I am examining inspections from January 2022 through October 2026 in both boroughs.

My preliminary analysis of data from the past five years shows that 24.2 percent of Bronx cooling tower inspections found at least one violation, compared with 17.6 percent in Manhattan. The disparity is sharper for the most serious category: 12.0 percent of Bronx inspections cited a Public Health Hazard, versus 7.7 percent in Manhattan, meaning a Bronx inspection was about 56 percent more likely to find one. 

As I continue, I will use graphs, tables, and maps, including neighborhood tabulation area, to examine where compliance gaps appear and if/how they intersect with socioeconomics.

_Note: Open Records data for 2026 inspections is complete through October 2026._

**Sources:** 

https://data.cityofnewyork.us/Health/NYC-Cooling-Tower-System-Inspection-Results/f9wb-g8mb/about_data

https://www.nyc.gov/site/doh/health/health-topics/legionnaires-disease.page

https://www.cbsnews.com/newyork/news/upper-east-side-legionnaires-disease-death-toll-august-2026/

https://evidence.nejm.org/doi/full/10.1056/EVIDpha2600062?referrer=https%3A%2F%2Fwww.google.com%2F


In [1]:
import glob
import pandas as pd

CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
print("Using:", CSV)

Using: NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv


To begin with, I am going to establish how many cooling towers there are listed in NYC Open Records for the Bronx and Manhattan using address and BIN number, a unique identifier assigned by the Dept. of  Buildings. This gives me a baseline for my data investigation.

In [2]:
import glob, os
import pandas as pd
import plotly.express as px
from IPython.display import display

# =====================================================================
# STEP 1: Load, then keep ONE row per cooling tower system (its latest record)
# =====================================================================
CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["Inspection_date"])
os.makedirs("charts", exist_ok=True)

systems = df.sort_values("date").groupby("System_ID").tail(1).copy()   # latest record per tower
print("Status of all towers (latest record):")
print(systems.groupby(["Borough", "Status"]).size().unstack(fill_value=0).to_string(), "\n")

inactive_counts = systems[systems["Status"] == "Inactive"].groupby("Borough").size()
systems = systems[systems["Status"] == "Active"].copy()     # building analysis uses ACTIVE towers only

# =====================================================================
# STEP 2: Decide what counts as one "building"
# BIN (Building Identification Number) is the city's official building ID.
# A few towers carry a placeholder BIN (1000000 or 2000000) meaning "unknown",
# so for those we fall back to the street address instead.
# =====================================================================
systems["address_clean"] = systems["Address"].str.upper().str.strip()
placeholder = systems["BIN"] % 1000000 == 0
systems["building_id"] = systems["BIN"].astype(int).astype(str)
systems.loc[placeholder, "building_id"] = "ADDR: " + systems.loc[placeholder, "address_clean"]

print(f"{len(systems):,} ACTIVE cooling tower systems; "
      f"{placeholder.sum()} had a placeholder BIN and were matched by address instead\n")

# =====================================================================
# STEP 3: Towers per building
# =====================================================================
per_building = (
    systems.groupby(["Borough", "building_id"])
    .agg(towers=("System_ID", "nunique"), address=("address_clean", "first"))
    .reset_index()
)

summary = per_building.groupby("Borough").agg(
    **{"Unique buildings (active towers)": ("building_id", "nunique"),
       "Active tower systems": ("towers", "sum"),
       "Avg active towers per building": ("towers", "mean"),
       "Most active towers in one building": ("towers", "max")}
)
summary["Avg active towers per building"] = summary["Avg active towers per building"].round(2)
summary["Inactive tower systems"] = inactive_counts
summary["All tower systems"] = summary["Active tower systems"] + summary["Inactive tower systems"]

summary = summary[["Unique buildings (active towers)", "Active tower systems",
                   "Inactive tower systems", "All tower systems",
                   "Avg active towers per building", "Most active towers in one building"]]

summary.loc["Total"] = [
    summary["Unique buildings (active towers)"].sum(),
    summary["Active tower systems"].sum(),
    summary["Inactive tower systems"].sum(),
    summary["All tower systems"].sum(),
    round(summary["Active tower systems"].sum() / summary["Unique buildings (active towers)"].sum(), 2),
    summary["Most active towers in one building"].max(),
]
for c in summary.columns:
    if c != "Avg active towers per building":
        summary[c] = summary[c].astype(int)

print("Cooling tower systems and buildings by borough")
display(summary)

# =====================================================================
# STEP 4: How many buildings have 1, 2, 3, or 4+ towers?
# =====================================================================
per_building["tower_group"] = per_building["towers"].clip(upper=4).map(
    {1: "1 tower", 2: "2 towers", 3: "3 towers", 4: "4+ towers"}
)
dist = (
    per_building.groupby(["Borough", "tower_group"]).size()
    .unstack(fill_value=0)[["1 tower", "2 towers", "3 towers", "4+ towers"]]
)
dist_pct = dist.div(dist.sum(axis=1), axis=0).mul(100).round(1)

print("\nNumber of buildings, by how many towers each has")
display(dist)
print("\nShare of each borough's buildings (%)")
display(dist_pct)

# =====================================================================
# STEP 5: Charts
# =====================================================================
plot_df = dist_pct.reset_index().melt(id_vars="Borough", var_name="Towers in building",
                                     value_name="% of buildings")
fig1 = px.bar(
    plot_df, x="Towers in building", y="% of buildings", color="Borough", barmode="group",
    color_discrete_map={"Bronx": "#d62728", "Manhattan": "#1f77b4"}, text="% of buildings",
    title="Active cooling towers per building: most buildings have just one",
)
fig1.update_layout(template="plotly_white")
fig1.write_html("charts/towers_per_building.html")
fig1.show()

top = per_building.sort_values("towers", ascending=False).head(10)
fig2 = px.bar(
    top.sort_values("towers"), x="towers", y="address", color="Borough", orientation="h",
    color_discrete_map={"Bronx": "#d62728", "Manhattan": "#1f77b4"},
    labels={"towers": "Cooling tower systems", "address": ""},
    title="Ten buildings with the most active cooling tower systems",
)
fig2.update_layout(template="plotly_white")
fig2.write_html("charts/top_buildings.html")
fig2.show()
display(top[["Borough", "address", "towers"]].reset_index(drop=True))

Status of all towers (latest record):
Status     Active  Inactive
Borough                    
Bronx         241        67
Manhattan    3703       768 

3,944 ACTIVE cooling tower systems; 5 had a placeholder BIN and were matched by address instead

Cooling tower systems and buildings by borough


,Unique buildings (active towers),Active tower systems,Inactive tower systems,All tower systems,Avg active towers per building,Most active towers in one building
Borough,,,,,,
Bronx,187,241,67,308,1.29,7
Manhattan,2938,3703,768,4471,1.26,18
Total,3125,3944,835,4779,1.26,18



Number of buildings, by how many towers each has


tower_group,1 tower,2 towers,3 towers,4+ towers
Borough,,,,
Bronx,157,18,6,6
Manhattan,2424,372,101,41



Share of each borough's buildings (%)


tower_group,1 tower,2 towers,3 towers,4+ towers
Borough,,,,
Bronx,84.0,9.6,3.2,3.2
Manhattan,82.5,12.7,3.4,1.4


,Borough,address,towers
0,Manhattan,32 6TH AVE,18
1,Manhattan,75 9TH AVE,13
2,Manhattan,60 EAST 42 STREET,10
3,Manhattan,111 8TH AVE,9
4,Manhattan,100 W 125TH ST,8
5,Manhattan,1501 BROADWAY,8
6,Manhattan,225 BROADWAY,8
7,Manhattan,111 5TH AVE,7
8,Manhattan,450 W 33RD ST,7
9,Bronx,111 EAST 210TH STREET,7


What the data above does is provide a headcount of the towers and the buildings that house them:

+ The Bronx has 241 active towers (plus 67 inactive), in 187 buildings.
+ Manhattan has 3,703 active towers (plus 768 inactive), in 2,938 buildings.
+ Most buildings in both boroughs have just one tower (84% in the Bronx, 82.5% in Manhattan).

This table is context. It shows how big each group is, which matters for how much weight each borough's inspection results carry. The Bronx has roughly one-fifteenth as many active towers as Manhattan.

Next up, I'm digging into the violations data.

In [3]:
import glob
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display

# ---------- STEP 0: load ----------
CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
print("Using:", CSV, "\n")

df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["Inspection_date"])
df["year"] = df["date"].dt.year
df["has_violation"] = df["Violation_Code"].notna()   # a violation = a row with a violation code

# ---------- FIGURE 1: dataset size ----------
total_rows = len(df)
violation_entries = df["has_violation"].sum()
print("FIGURE 1: dataset size")
print(f"  Total rows in file:            {total_rows:,}")
print(f"  Rows with a violation:         {violation_entries:,}")
print(f"  Rows with no violation cited:  {total_rows - violation_entries:,}")
print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# ---------- FIGURE 2: collapse to one row per inspection ----------
inspections = (
    df.groupby(["System_ID", "Inspection_date", "Borough", "year", "Inspection_Type"])
    .agg(
        any_violation=("has_violation", "max"),
        phh=("Violation_Type", lambda s: (s == "PHH").any()),
    )
    .reset_index()
)
print("\nFIGURE 2: inspections")
print(f"  Unique inspections: {len(inspections):,}")

# ---------- FIGURES 3-4: rates by borough ----------
START, END = 2022, 2026          # change to (2022, 2025) to exclude 2026
window = inspections[inspections["year"].between(START, END)]

table = window.groupby("Borough").agg(
    inspections=("any_violation", "size"),
    with_violation=("any_violation", "sum"),
    with_phh=("phh", "sum"),
)
table["pct_any_violation"] = (table["with_violation"] / table["inspections"] * 100).round(1)
table["pct_phh"] = (table["with_phh"] / table["inspections"] * 100).round(1)

table = table.rename(columns={
    "inspections": "Inspections",
    "with_violation": "With a Violation",
    "with_phh": "With a Public Health Hazard",
    "pct_any_violation": "% with a Violation",
    "pct_phh": "% with a Public Health Hazard",
})
table.index.name = "Borough"

print(f"\nFIGURES 3-4: {START}-{END} rates")
display(table.style.format({
    "Inspections": "{:,}",
    "With a Violation": "{:,}",
    "With a Public Health Hazard": "{:,}",
    "% with a Violation": "{:.1f}",
    "% with a Public Health Hazard": "{:.1f}",
}))

# ---------- Significance tests ----------
print("Significance tests (chi-square):")
for col, label in [("any_violation", "any violation"), ("phh", "public health hazard")]:
    p = chi2_contingency(pd.crosstab(window["Borough"], window[col]))[1]
    print(f"  {label}: p = {p:.2g}")

Using: NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv 

FIGURE 1: dataset size
  Total rows in file:            95,893
  Rows with a violation:         67,975
  Rows with no violation cited:  27,918
  Date range: 2017-01-17 to 2026-10-16

FIGURE 2: inspections
  Unique inspections: 29,450

FIGURES 3-4: 2022-2026 rates


,Inspections,With a Violation,With a Public Health Hazard,% with a Violation,% with a Public Health Hazard
Borough,,,,,
Bronx,"1,064",257,128,24.2,12.0
Manhattan,"11,193","1,966",862,17.6,7.7


Significance tests (chi-square):
  any violation: p = 1.2e-07
  public health hazard: p = 9.9e-07


_The difference between the boroughs is statistically significant; it's very unlikely to be due to chance._

The table above shows that between 2022 and October 2026, a greater share of inspections in the Bronx (24.2%) found at least one violation than in Manhattan (17.6%). Because Manhattan had about ten times as many inspections (11,193 versus 1,064), I compare percentages rather than raw counts.

The difference is also visible at the most serious level. In the Bronx, 12.0% of inspections cited at least one Public Health Hazard, which the Department of Health classifies as its most serious violation type, compared with 7.7% in Manhattan.

These figures count each inspection once, whether it found one violation or several. To show how many violations that inspections turned up, I'm now going to look at violations issued. I'll compare the two boroughs on Critical violations, one level below Public Health Hazard, and General violations, the least serious type. Because the boroughs were inspected at very different rates, I measure each as violations per 100 inspections and calculate the percentage difference between the two.

In [4]:
import glob
import pandas as pd
from IPython.display import display

# ---------- Load ----------
CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
df = pd.read_csv(CSV)
df["year"] = pd.to_datetime(df["Inspection_date"]).dt.year

START, END = 2022, 2026
window = df[df["year"].between(START, END)]

# ---------- Violations by type and borough ----------
viol = window[window["Violation_Type"].notna()]
counts = pd.crosstab(viol["Borough"], viol["Violation_Type"])[["General", "Critical", "PHH"]]
counts.columns = ["General", "Critical", "Public health hazard"]
counts["All types"] = counts.sum(axis=1)

# Unique inspections per borough
n_insp = (
    window.drop_duplicates(["System_ID", "Inspection_date", "Inspection_Type"])
    .groupby("Borough").size()
)
per100 = counts.div(n_insp, axis=0) * 100

# ---------- Comparison table ----------
comparison = pd.DataFrame({
    "Bronx violations": counts.loc["Bronx"],
    "Manhattan violations": counts.loc["Manhattan"],
    "Bronx per 100 inspections": per100.loc["Bronx"],
    "Manhattan per 100 inspections": per100.loc["Manhattan"],
})
comparison["% difference (Bronx vs. Manhattan)"] = (
    (comparison["Bronx per 100 inspections"] - comparison["Manhattan per 100 inspections"])
    / comparison["Manhattan per 100 inspections"] * 100
)
comparison.index.name = "Violation type"

print(f"Violations issued, {START}-{END}")
display(comparison.style.format({
    "Bronx violations": "{:,.0f}",
    "Manhattan violations": "{:,.0f}",
    "Bronx per 100 inspections": "{:.1f}",
    "Manhattan per 100 inspections": "{:.1f}",
    "% difference (Bronx vs. Manhattan)": "{:+.1f}%",
}))


Violations issued, 2022-2026


,Bronx violations,Manhattan violations,Bronx per 100 inspections,Manhattan per 100 inspections,% difference (Bronx vs. Manhattan)
Violation type,,,,,
General,288,"1,414",27.1,12.6,+114.3%
Critical,576,"3,313",54.1,29.6,+82.9%
Public health hazard,160,"1,042",15.0,9.3,+61.5%
All types,"1,024","5,769",96.2,51.5,+86.7%


For every 100 inspections, Bronx cooling towers were cited for about 96 violations, compared with about 52 in Manhattan. Measured against Manhattan's rate, the Bronx's was 87 percent higher across all violation types, and it was higher for each type individually: General violations (+114 percent), Critical violations (+83 percent), and Public Health Hazard violations (+62 percent).

The graph below shows these rates by violation type.

In [5]:
import glob, os
import pandas as pd
import plotly.graph_objects as go

# ---------- Load ----------
CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
df = pd.read_csv(CSV)
df["year"] = pd.to_datetime(df["Inspection_date"]).dt.year
os.makedirs("charts", exist_ok=True)

START, END = 2022, 2026
window = df[df["year"].between(START, END)]

# ---------- Violations by type and borough ----------
viol = window[window["Violation_Type"].notna()]
counts = pd.crosstab(viol["Borough"], viol["Violation_Type"])[["General", "Critical", "PHH"]]
counts.columns = ["General", "Critical", "Public health hazard"]

n_insp = (
    window.drop_duplicates(["System_ID", "Inspection_date", "Inspection_Type"])
    .groupby("Borough").size()
)
per100 = counts.div(n_insp, axis=0) * 100
pct_diff = (per100.loc["Bronx"] - per100.loc["Manhattan"]) / per100.loc["Manhattan"] * 100

total_bronx = counts.loc["Bronx"].sum() / n_insp["Bronx"] * 100
total_manh = counts.loc["Manhattan"].sum() / n_insp["Manhattan"] * 100
total_diff = (total_bronx - total_manh) / total_manh * 100

# ---------- Chart ----------
COLORS = {"Bronx": "#d62728", "Manhattan": "#1f77b4"}
types = list(per100.columns)

fig = go.Figure()
for boro in ["Bronx", "Manhattan"]:
    fig.add_trace(go.Bar(
        x=types, y=per100.loc[boro].round(1), name=boro,
        marker_color=COLORS[boro],
        text=per100.loc[boro].round(1), textposition="inside",
        textfont=dict(color="white", size=14),
    ))

# Percentage-difference labels above each pair of bars
for t in types:
    fig.add_annotation(
        x=t, y=per100[t].max() + 4, text=f"<b>{pct_diff[t]:+.1f}%</b>",
        showarrow=False, font=dict(size=15),
    )

fig.update_layout(
    title=dict(
        text=(f"Violations per 100 inspections by type, {START}-{END}"
              f"<br><sup>Labels show how much higher the Bronx rate is than Manhattan's. "
              f"All types: {total_bronx:.1f} vs. {total_manh:.1f} ({total_diff:+.1f}%)</sup>")
    ),
    barmode="group", template="plotly_white", height=500,
    yaxis_title="Violations per 100 inspections",
    yaxis_range=[0, per100.values.max() + 12],
    legend_title_text="",
)
fig.write_html("charts/violations_per_100_by_type.html")
fig.show()

More to come! 